# Feature Engineering

<a target="_blank" href="https://colab.research.google.com/github/imamitjain/notebooks/blob/main/02-ml-fundamentals/04_feature_engineering.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective:** Transform raw data into effective features — encoding categoricals, scaling numerics, creating interactions, and selecting the most informative features.

**Prerequisites:** Clustering and unsupervised (notebook 03)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q numpy pandas matplotlib scikit-learn


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

## 1. Encoding Categorical Variables

In [ ]:
df = pd.DataFrame({
    'color': ['red', 'blue', 'green', 'red', 'blue'],
    'size': ['S', 'M', 'L', 'XL', 'M'],
    'price': [10, 20, 30, 15, 25]
})

# Label encoding (ordinal)
le = LabelEncoder()
df['size_encoded'] = le.fit_transform(df['size'])
print("Label encoded sizes:")
print(df[['size', 'size_encoded']])

# One-hot encoding
ohe = OneHotEncoder(sparse_output=False)
color_encoded = ohe.fit_transform(df[['color']])
color_df = pd.DataFrame(color_encoded, columns=ohe.get_feature_names_out())
print(f"\nOne-hot encoded colors:\n{color_df}")

## 2. Scaling and Normalization

In [ ]:
rng = np.random.default_rng(42)
data = pd.DataFrame({
    'age': rng.integers(18, 80, 100),
    'income': rng.integers(20000, 200000, 100),
    'score': rng.uniform(0, 1, 100)
})

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].boxplot(data.values, labels=data.columns)
axes[0].set_title('Original (different scales)')

standard = StandardScaler().fit_transform(data)
axes[1].boxplot(standard, labels=data.columns)
axes[1].set_title('StandardScaler (mean=0, std=1)')

minmax = MinMaxScaler().fit_transform(data)
axes[2].boxplot(minmax, labels=data.columns)
axes[2].set_title('MinMaxScaler (0-1)')

plt.tight_layout()
plt.show()

## 3. Polynomial and Interaction Features

In [ ]:
X_simple = np.array([[1, 2], [3, 4], [5, 6]])
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_simple)

print("Original features: [x1, x2]")
print(f"Polynomial features: {poly.get_feature_names_out()}")
print(f"\nOriginal:\n{X_simple}")
print(f"\nWith interactions:\n{X_poly}")

## 4. Feature Selection (Filter and Model-Based)

In [ ]:
from sklearn.datasets import make_classification

X_fs, y_fs = make_classification(n_samples=200, n_features=20, n_informative=5,
                                  n_redundant=5, random_state=42)

# Filter method: SelectKBest
selector = SelectKBest(f_classif, k=10)
X_selected = selector.fit_transform(X_fs, y_fs)

scores = selector.scores_
plt.figure(figsize=(10, 4))
plt.bar(range(20), scores, color=['steelblue' if s else 'lightgray'
        for s in selector.get_support()])
plt.xlabel('Feature Index')
plt.ylabel('F-Score')
plt.title('Feature Importance (SelectKBest)')
plt.show()

# Model-based: Random Forest importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_fs, y_fs)
importances = rf.feature_importances_

plt.figure(figsize=(10, 4))
plt.bar(range(20), importances)
plt.xlabel('Feature Index')
plt.ylabel('Importance')
plt.title('Random Forest Feature Importance')
plt.show()

## 5. Building a Full Pipeline with ColumnTransformer

In [ ]:
df_mixed = pd.DataFrame({
    'age': [25, 30, 35, 40, 28, 32, 45, 22, 38, 50],
    'income': [50000, 60000, 75000, 90000, 55000, 65000, 100000, 45000, 80000, 95000],
    'education': ['BS', 'MS', 'PhD', 'BS', 'MS', 'BS', 'PhD', 'BS', 'MS', 'PhD'],
    'city': ['NYC', 'LA', 'NYC', 'Chicago', 'LA', 'NYC', 'Chicago', 'LA', 'NYC', 'Chicago'],
    'target': [0, 1, 1, 1, 0, 0, 1, 0, 1, 1]
})

numeric_features = ['age', 'income']
categorical_features = ['education', 'city']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
])

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

X_mixed = df_mixed.drop('target', axis=1)
y_mixed = df_mixed['target']

scores = cross_val_score(pipe, X_mixed, y_mixed, cv=3, scoring='accuracy')
print(f"Pipeline CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

pipe.fit(X_mixed, y_mixed)
feature_names = (numeric_features +
                 list(pipe.named_steps['preprocessor']
                      .named_transformers_['cat'].get_feature_names_out()))
print(f"\nTransformed features: {feature_names}")

## Try It Yourself

1. Take a dataset with mixed types (numeric + categorical). Build a `ColumnTransformer` pipeline that scales numerics and one-hot encodes categoricals, then train a random forest.
2. Compare model performance with raw features vs. polynomial features (degree 2).
3. Use `SelectKBest` with mutual information to pick the top 5 features from a 20-feature dataset. Does the reduced model match the full model?

In [ ]:
# Your code here